# DEAP Dataset — Exploratory Data Analysis

Этот ноутбук только загружает датасет и визуализирует сигналы и метки.  
**Никакого обучения нет.** Достаточно установить зависимости и указать путь к данным.

**Требования:**
```
pip install numpy scipy matplotlib
```

## 0. Настройка пути к данным

In [ ]:
from pathlib import Path

# ← УКАЖИ ПУТЬ К ПАПКЕ data_preprocessed_python
DATA_DIR = Path(r"C:\Users\psgpe\Downloads\2025-2026_NeuroBarometer Emotions\2025-2026_NeuroBarometer Emotions\DEAP\dataset\data_preprocessed_python")

assert DATA_DIR.exists(), f"Папка не найдена: {DATA_DIR}"
files = sorted(DATA_DIR.glob("s*.dat"))
print(f"Найдено файлов: {len(files)}")
print("Первые 5:", [f.name for f in files[:5]])

## 1. Загрузка одного субъекта

In [ ]:
import pickle
import numpy as np

SUBJECT_ID = 1   # ← меняй от 1 до 32

with open(DATA_DIR / f"s{SUBJECT_ID:02d}.dat", "rb") as f:
    raw = pickle.load(f, encoding="latin1")

data   = raw["data"].astype(np.float64)    # (40, 40, 8064)
labels = raw["labels"].astype(np.float32)  # (40, 4)

print(f"data shape  : {data.shape}    — (trials, channels, samples)")
print(f"labels shape: {labels.shape}  — (trials, [valence, arousal, dominance, liking])")
print(f"\nПервые 5 меток (valence, arousal, dom, liking):")
print(labels[:5])

In [ ]:
SFREQ           = 128
BASELINE_SAMPLES = 384   # первые 3 сек

# Разделяем базовую линию и стимульный сигнал
baseline = data[:, :, :BASELINE_SAMPLES]   # (40, 40, 384)
trial    = data[:, :, BASELINE_SAMPLES:]   # (40, 40, 7680)

# Модальности
EEG_CH = list(range(32))
GSR_CH = 36
PPG_CH = 38

eeg = trial[:, EEG_CH, :]   # (40, 32, 7680)
gsr = trial[:, GSR_CH, :]   # (40, 7680)
ppg = trial[:, PPG_CH, :]   # (40, 7680)

print(f"EEG : {eeg.shape}  ({eeg.shape[2]/SFREQ:.0f} с × {eeg.shape[1]} каналов)")
print(f"GSR : {gsr.shape}")
print(f"PPG : {ppg.shape}")

## 2. Сигналы одного trial

In [ ]:
import matplotlib.pyplot as plt

TRIAL_IDX = 0   # ← меняй от 0 до 39
t = np.arange(7680) / SFREQ  # временная ось, сек

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

# EEG — первые 4 канала
for ch in range(4):
    axes[0].plot(t, eeg[TRIAL_IDX, ch], lw=0.6, alpha=0.8, label=f"ch{ch}")
axes[0].set_title(f"EEG (первые 4 канала) — trial {TRIAL_IDX}")
axes[0].set_ylabel("мкВ")
axes[0].legend(ncol=4, fontsize=8)

# EEG — все 32 канала (усреднённые)
axes[1].plot(t, eeg[TRIAL_IDX].mean(axis=0), lw=0.8, color="#4472C4")
axes[1].set_title("EEG среднее по 32 каналам")
axes[1].set_ylabel("мкВ")

# PPG
axes[2].plot(t, ppg[TRIAL_IDX], lw=0.8, color="#ED7D31")
axes[2].set_title("PPG / BVP")
axes[2].set_ylabel("а.е.")

# GSR
axes[3].plot(t, gsr[TRIAL_IDX], lw=0.8, color="#70AD47")
axes[3].set_title("GSR / EDA")
axes[3].set_ylabel("мкСм")
axes[3].set_xlabel("Время (с)")

v, a = labels[TRIAL_IDX, 0], labels[TRIAL_IDX, 1]
fig.suptitle(f"s{SUBJECT_ID:02d} — Trial {TRIAL_IDX} | Valence={v:.1f}  Arousal={a:.1f}", fontsize=13)
plt.tight_layout()
plt.show()

## 3. Распределение меток субъекта

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, name, color in zip(axes, [0, 1], ["Valence", "Arousal"], ["#4472C4", "#ED7D31"]):
    vals = labels[:, col]
    median = np.median(vals)
    ax.hist(vals, bins=9, range=(1, 10), color=color, alpha=0.8, edgecolor="white")
    ax.axvline(median, color="black", linestyle="--", lw=2, label=f"Медиана = {median:.1f}")
    n_high = (vals > median).sum()
    n_low  = (vals <= median).sum()
    ax.set_title(f"{name} | High: {n_high}  Low: {n_low}")
    ax.set_xlabel("Оценка [1–9]")
    ax.set_ylabel("Trials")
    ax.legend()
    ax.set_xlim(1, 9)

fig.suptitle(f"Распределение меток — s{SUBJECT_ID:02d}", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Метки по всем 32 субъектам

In [ ]:
print("Загружаем метки всех субъектов (только labels, без сигналов — быстро)...")
all_labels = {}
for sid in range(1, 33):
    path = DATA_DIR / f"s{sid:02d}.dat"
    if not path.exists():
        continue
    with open(path, "rb") as f:
        r = pickle.load(f, encoding="latin1")
    all_labels[sid] = r["labels"].astype(np.float32)
    print(f"  s{sid:02d}", end="" if sid % 8 else "\n")
print(f"\nЗагружено {len(all_labels)} субъектов")

In [ ]:
# Средние оценки по видео (консенсус)
stacked = np.stack([all_labels[s] for s in sorted(all_labels)], axis=0)  # (32, 40, 4)
mean_per_video = stacked.mean(axis=0)  # (40, 4)  среднее по субъектам

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
videos = np.arange(1, 41)

for ax, col, name, color in zip(axes, [0, 1], ["Valence", "Arousal"], ["#4472C4", "#ED7D31"]):
    means = mean_per_video[:, col]
    stds  = stacked[:, :, col].std(axis=0)
    ax.bar(videos, means, color=color, alpha=0.75, label="Среднее")
    ax.errorbar(videos, means, yerr=stds, fmt="none", color="black", capsize=2, lw=0.8)
    ax.axhline(5, color="gray", linestyle=":", lw=1.2)
    ax.set_xlabel("Видео (trial index)")
    ax.set_ylabel("Оценка [1–9]")
    ax.set_title(f"{name} — среднее по всем субъектам (±std)")
    ax.set_ylim(1, 9)

fig.suptitle("Межсубъектный консенсус по 40 видео", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Видео с наивысшей Valence: trial {mean_per_video[:,0].argmax()} ({mean_per_video[:,0].max():.2f})")
print(f"Видео с наивысшим Arousal: trial {mean_per_video[:,1].argmax()} ({mean_per_video[:,1].max():.2f})")

## 5. Спектр мощности EEG (один trial)

In [ ]:
from scipy.signal import welch

TRIAL_IDX = 0
freqs, psd = welch(eeg[TRIAL_IDX], fs=SFREQ, nperseg=256, axis=-1)  # (32, n_freqs)
mean_psd = psd.mean(axis=0)  # среднее по каналам

bands = {"delta": (1,3), "theta": (4,7), "alpha": (8,13), "beta": (14,30), "gamma": (31,50)}
colors = ["#5B9BD5", "#70AD47", "#FFC000", "#FF0000", "#7030A0"]

fig, ax = plt.subplots(figsize=(12, 4))
mask = freqs <= 50
ax.semilogy(freqs[mask], mean_psd[mask], color="#333333", lw=1.5)

for (name, (lo, hi)), col in zip(bands.items(), colors):
    bm = (freqs >= lo) & (freqs <= hi)
    ax.fill_between(freqs[bm], mean_psd[bm], alpha=0.4, color=col, label=name)

ax.set_xlabel("Частота (Гц)")
ax.set_ylabel("PSD (мкВ²/Гц, log)")
ax.set_title(f"Спектр мощности EEG — s{SUBJECT_ID:02d}, trial {TRIAL_IDX} (среднее 32 каналов)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Topographic map (EEG по каналам)

In [ ]:
# Средняя мощность в alpha-полосе по каналам (один trial)
from scipy.signal import welch

freqs, psd_all = welch(eeg[TRIAL_IDX], fs=SFREQ, nperseg=256, axis=-1)  # (32, n_freqs)
alpha_mask = (freqs >= 8) & (freqs <= 13)
alpha_power = psd_all[:, alpha_mask].mean(axis=1)  # (32,)

# Позиции каналов 10-20 (упрощённо для 32 каналов DEAP)
ch_names_32 = [
    "Fp1","AF3","F7","F3","FC1","FC5","T7","C3","CP1","CP5",
    "P7","P3","Pz","PO3","O1","Oz","O2","PO4","P4","P8",
    "CP6","CP2","C4","T8","FC6","FC2","F4","F8","AF4","Fp2",
    "Fz","Cz"
]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(32)
bars = ax.bar(x, alpha_power, color=plt.cm.RdYlBu_r(alpha_power / alpha_power.max()))
ax.set_xticks(x)
ax.set_xticklabels(ch_names_32, rotation=90, fontsize=8)
ax.set_ylabel("Alpha мощность (мкВ²/Гц)")
ax.set_title(f"Alpha мощность по каналам — s{SUBJECT_ID:02d}, trial {TRIAL_IDX}")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()